In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
%matplotlib inline


torch.manual_seed(12046)

In [ ]:
class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        '''
        长短期记忆网络的神经元
        参数
        ---
        input_size:int,输入数据的特征长度
        hidden_size:int,隐藏状态的特征长度
        '''
        super(LSTMCell, self).__init__()
        self.input_size = input_size  # input_size :I
        self.hidden_size = hidden_size  # hidden_size :H
        combined_size = self.input_size + self.hidden_size  # combined_size :I+H
        # 定义输入们的线性部分
        self.in_gate = nn.Linear(combined_size, self.hidden_size)
        # 定义遗忘门的线性部分
        self.forget_gate = nn.Linear(combined_size, self.hidden_size)
        # 定义备选细胞状态的线性部分
        self.new_cell_gate = nn.Linear(combined_size, self.hidden_size)
        # 定义输入门的线性部分
        self.out_gate = nn.Linear(combined_size, self.hidden_size)

    def forward(self, inputs, state=None):
        '''
        向前传播
        参数
        ---
        inputs:torch.FloatTensor,输入数据，形状为(B,I),其中B表示批量大小，I表示文字特征的长度(input_size)
        state ：tuple(torch.FloatTensor, torch.FloatTensor)
            (隐藏状态，细胞状态)，两个状态的形状都为(B, H)，其中H表示隐藏状态的长度（hidden_size）
        返回
        ----
        hs ：torch.FloatTensor，隐藏状态，形状为(B, H)
        cs ：torch.FloatTensor，细胞状态，形状为(B, H)
        '''
        B, _ = inputs.shape
        if state is None:
            state = self.init_state(B, inputs.device)
        hs, cs = state
        combined = torch.cat((inputs, hs), dim=1)  # (B,I+H)
        # 输入门
        ingate = F.sigmoid(self.in_gate(combined))  # (B,    H)
        # 遗忘门
        forgetgate = F.sigmoid(self.forget_gate(combined))  # (B,     H)
        # 输入门
        outgate = F.sigmoid(self.out_gate(combined))  # (B,     H)
        # 更新细胞状态
        ncs = F.tanh(self.new_cell_gate(combined))
        cs = (forgetgate * cs) + (ingate * ncs)
        # 更新隐藏状态
        hs = outgate * F.tanh(cs)
        return hs, cs

    def init_state(self, B, device):
        # 默认的隐藏状态和细胞状态全部都等于0
        cs=torch.zeros((B,self.hidden_size),device=device)
        hs=torch.zeros((B,self.hidden_size),device=device)
        return hs,cs

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        '''
        单层的长短期记忆网络（支持批量计算）
        参数
        ---
        input_size:int,输入数据的特征长度
        hidden_size:int,隐藏状态的特征长度
        '''
        super(LSTM, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.lstm = LSTMCell(self.input_size, self.hidden_size)

    def forward(self, inputs, state=None):
        '''
        向前传播
        参数
        ----
        inputs ：torch.FloatTensor
            输入数据的集合，形状为(B, T, C)，其中B表示批量大小，T表示文本长度，C表示文字特征的长度（input_size）
        state ：tuple(torch.FloatTensor, torch.FloatTensor)
            (初始的隐藏状态，初始的细胞状态)，两个状态的形状都为(B, H)，其中H表示隐藏状态的长度（hidden_size）
        返回
        ----
        hidden ：torch.FloatTensor，所有隐藏状态的集合，形状为(B, T, H)
        '''
        re = []
        B, T, C = inputs.shape
        inputs = inputs.transpose(0, 1)  # (T,B,C)
        for i in range(T):
            state = self.lstm(inputs[i], state)
            # 只记录隐藏状态，state[0]的形状为(B,H)
            re.append(state[0])
        result_tensor=torch.stack(re,dim=0) # (T,B,H)
        return result_tensor.transpose(0,1) # (B,T,H)

In [ ]:
def test_lstm():
    '''
    测试LSTM实现的准确性
    '''
    # 随机生成模型结构
    B, T, input_size, hidden_size, num_layers = torch.randint(1, 20, (5,)).tolist()
    ref_model = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
    # 随机生成输入
    inputs = torch.randn(B, T, input_size)
    hs, cs = torch.randn((2 * num_layers, B, hidden_size)).chunk(2, 0)
    re = inputs
    # 取出模型参数
    for layer_index in range(num_layers):
        l = ref_model.all_weights[layer_index]
        if layer_index == 0:
            model = LSTM(input_size, hidden_size)
        else:
            model = LSTM(hidden_size, hidden_size)
        i, f, c, o = torch.cat((l[0], l[1]), dim=1).chunk(4, 0)
        ib, fb, cb, ob = (l[2] + l[3]).chunk(4, 0)
        # 设置模型参数
        model.lstm.in_gate.weight = nn.Parameter(i)
        model.lstm.in_gate.bias = nn.Parameter(ib)
        model.lstm.forget_gate.weight = nn.Parameter(f)
        model.lstm.forget_gate.bias = nn.Parameter(fb)
        model.lstm.new_cell_gate.weight = nn.Parameter(c)
        model.lstm.new_cell_gate.bias = nn.Parameter(cb)
        model.lstm.out_gate.weight = nn.Parameter(o)
        model.lstm.out_gate.bias = nn.Parameter(ob)
        # 计算隐藏状态
        re = model(re, (hs[layer_index], cs[layer_index]))
    ref_re, _ = ref_model(inputs, (hs, cs))
    # 验证计算结果（最后一层的隐藏状态是否一致）
    out = torch.all(torch.abs(re - ref_re) < 1e-4)
    return out, (B, T, input_size, hidden_size, num_layers)

test_lstm()

In [ ]:
# 一些超参数
learning_rate = 1e-3
eval_iters = 10
batch_size=1000
sequence_len=64
# 如果有GPU，该脚本将使用GPU进行计算
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
raw_datasets = load_dataset("code_search_net", "python")
datasets = raw_datasets['train'].filter(lambda x: 'apache/spark' in x['repository_name'])

class char_tokenizer:

    def __init__(self, data):
        # 数据中出现的所有字符构成字典
        chars = sorted(list(set(''.join(data))))
        # 预留一个位置给结尾的特殊字符
        self.char2ind = {s : i + 1 for i, s in enumerate(chars)}
        self.char2ind['<|e|>'] = 0
        self.ind2char = {i : s for s, i in self.char2ind.items()}

    def encode(self, text):
        return [self.char2ind[c] for c in text]

    def decode(self, enc):
        if isinstance(enc, int):
            return self.ind2char[enc]
        return [self.ind2char[i] for i in enc]

tok = char_tokenizer(datasets['whole_func_string'])
len(tok.char2ind)

In [ ]:
class CharLSTM(nn.Module):

    def __init__(self, vs):
        '''
        三层的长短期记忆网络
        参数
        ----
        vs ：int，字典大小
        '''
        super().__init__()
        # 定义文字嵌入的特征长度
        self.emb_size = 256
        # 定义隐藏状态的特征长度
        self.hidden_size = 128
        # 文字嵌入层
        self.embedding = nn.Embedding(vs, self.emb_size)
        # 随机失活
        self.dp = nn.Dropout(0.4)
        # 第一层长短期记忆网络
        self.lstm1 = LSTM(self.emb_size, self.hidden_size)
        # 层归一化
        self.norm1 = nn.LayerNorm(self.hidden_size)
        self.lstm2 = LSTM(self.hidden_size, self.hidden_size)
        self.norm2 = nn.LayerNorm(self.hidden_size)
        self.lstm3 = LSTM(self.hidden_size, self.hidden_size)
        self.norm3 = nn.LayerNorm(self.hidden_size)
        # 语言建模头，根据最后一层的隐藏状态预测下一个字母是什么
        self.h2o = nn.Linear(self.hidden_size, vs)

    def forward(self, x):
        '''
        向前传播
        参数
        ----
        x ：torch.LongTensor，当前字母在字典中的位置，形状为(B, T)
        返回
        ----
        output ：torch.FloatTensor，预测结果的logits，形状为(B, T, vs)
        '''
        emb = self.embedding(x)                   # (B, T,  C)
        h = self.norm1(self.dp(self.lstm1(emb)))  # (B, T,  H)
        # 第一层的隐藏状态是第二层的输入
        h = self.norm2(self.dp(self.lstm2(h)))    # (B, T,  H)
        # 第二层的隐藏状态是第三层的输入
        h = self.norm3(self.dp(self.lstm3(h)))    # (B, T,  H)
        # 使用第三层的隐藏状态预测下一个字母是什么
        output = self.h2o(h)                      # (B, T, vs)
        return output

model = CharLSTM(len(tok.char2ind)).to(device)

In [ ]:
# 展示模型结构
model

In [ ]:
@torch.no_grad()
def generate_batch(model, idx, max_new_tokens=300):
    '''
    利用模型生成文本（反复使用模型进行预测）
    参数
    ----
    model ：CharLSTM，生成文本的模型
    idx ：torch.LongTensor，当前字母在字典中的位置，形状为(1, T)
    max_new_tokens ：int，生成文本的最大长度
    返回
    ----
    out ：list[int]，生成的文本
    '''
    # 将模型切换至评估模式
    model.eval()
    for _ in range(max_new_tokens):
        # 限制背景长度，使之与模型训练时的状况更相符
        # 当然也可以不限制
        context = idx[:, -sequence_len:]
        # 在文本生成时，模型的计算效率很低，因为有很多重复计算
        logits = model(context)
        # 只使用最后一个预测结果
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        # 根据模型预测的概率，得到最终的预测结果（下一个字母）
        # 这一步运算有一定随机性
        ix = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, ix), dim=1)
        if ix.item() == 0:
            break
    # 将模型切换至训练模式
    model.train()
    return idx.tolist()[0]

In [ ]:
# 使用模型来生成文本
begin_text = torch.tensor(tok.encode('def'), device=device).unsqueeze(0)
print(''.join(tok.decode(generate_batch(model, begin_text))))

In [ ]:
def process(data, sequence_len=sequence_len):
    '''
    根据文本生成训练数据
    '''
    # text是字符串列表
    text = data['whole_func_string']
    inputs, labels = [], []
    for i in text:
        enc = tok.encode(i)
        # 0对应着文本结束
        enc += [0]
        # 将文本转换为多个训练数据
        for i in range(len(enc) - sequence_len):
            inputs.append(enc[i: i + sequence_len])
            # 预测标签是下一个字母，因此只需要挪动一个位置即可
            labels.append(enc[i + 1: i + 1 + sequence_len])
    return {'inputs': inputs, 'labels': labels}

# 将数据分为训练集和测试集
tokenized = datasets.train_test_split(test_size=0.1, seed=1024, shuffle=True)
# 将文本转换为训练数据，里面包含inputs和labels
tokenized = tokenized.map(process, batched=True, remove_columns=datasets.column_names)
tokenized.set_format(type='torch', device=device)

tokenized['train']['inputs'].shape, tokenized['train']['labels'].shape

In [ ]:
# 构建数据读取器
train_loader = DataLoader(tokenized['train'], batch_size=batch_size, shuffle=True)
test_loader = DataLoader(tokenized['test'], batch_size=batch_size, shuffle=True)
# 获取一个批量的数据
next(iter(test_loader))

In [ ]:
def estimate_loss(model):
    re = {}
    # 将模型切换至评估模式
    model.eval()
    re['train'] = _loss(model, train_loader)
    re['test'] = _loss(model, test_loader)
    # 将模型切换至训练模式
    model.train()
    return re

@torch.no_grad()
def _loss(model, data_loader):
    """
    计算模型在不同数据集下面的评估指标
    """
    loss = []
    data_iter= iter(data_loader)
    # 随机使用多个批量数据来预估模型效果
    for k in range(eval_iters):
        data = next(data_iter, None)
        if data is None:
            data_iter = iter(data_loader)
            data = next(data_iter, None)
        inputs, labels = data['inputs'], data['labels']
        logits = model(inputs)
        # 根据cross_entropy的定义，需要对logits进行转置运算
        # 具体细节请参考cross_entropy的官方文档
        logits = logits.transpose(-2, -1)
        loss.append(F.cross_entropy(logits, labels).item())
    return torch.tensor(loss).mean().item()

estimate_loss(model)

In [ ]:
def train_lstm(model, optimizer, data_loader, epochs=3):
    lossi = []
    for epoch in range(epochs):
        for i, data in enumerate(data_loader, 0):
            inputs, labels = data['inputs'], data['labels']
            optimizer.zero_grad()
            logits = model(inputs)
            # 根据cross_entropy的定义，需要对logits进行转置运算
            # 具体细节请参考cross_entropy的官方文档
            logits = logits.transpose(-2, -1)
            loss = F.cross_entropy(logits, labels)
            lossi.append(loss.item())
            loss.backward()
            optimizer.step()
        # 评估模型，并输出结果
        stats = estimate_loss(model)
        train_loss = f'train loss {stats["train"]:.4f}'
        test_loss = f'test loss {stats["test"]:.4f}'
        print(f'epoch {epoch:>2}: {train_loss}, {test_loss}')
    return lossi

In [ ]:
l = train_lstm(model, optim.Adam(model.parameters(), lr=learning_rate), train_loader)

In [ ]:
plt.plot(torch.tensor(l).view(-1, 10).mean(1).numpy())

In [ ]:
# 使用模型来生成文本
begin_text = torch.tensor(tok.encode('def'), device=device).unsqueeze(0)
print(''.join(tok.decode(generate_batch(model, begin_text))))

In [ ]:
# 将层归一化放到在LSTM神经元里面
class LSTMLayerNormCell(nn.Module):

    def __init__(self, input_size, hidden_size):
        '''
        长短期记忆网络的神经元（内含层归一化）
        参数
        ----
        input_size ：int，输入数据的特征长度
        hidden_size ：int，隐藏状态的特征长度
        '''
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        combined_size = self.input_size + self.hidden_size
        # 将四个线性模块放在一起定义，使得代码更加简洁和高效
        self.gates = nn.Linear(
            combined_size, 4 * self.hidden_size, bias=False)
        # 用于门的层归一化
        self.ln_gates = nn.LayerNorm(4 * self.hidden_size)
        # 用于细胞状态的层归一化
        self.ln_c = nn.LayerNorm(self.hidden_size)

    def forward(self, inputs, state=None):
        '''
        向前传播
        参数
        ----
        inputs ：torch.FloatTensor
            输入数据，形状为(B, I)，其中B表示批量大小，I表示文字特征的长度（input_size）
        state ：tuple(torch.FloatTensor, torch.FloatTensor)
            (隐藏状态，细胞状态)，两个状态的形状都为(B, H)，其中H表示隐藏状态的长度（hidden_size）
        返回
        ----
        hs ：torch.FloatTensor，隐藏状态，形状为(B, H)
        cs ：torch.FloatTensor，细胞状态，形状为(B, H)
        '''
        B, _ = inputs.shape
        if state is None:
            state = self.init_state(B, inputs.device)
        hs, cs = state
        combined = torch.cat((inputs, hs), dim=1)  # (B, I + H)
        # 将四个线性模块分开
        i, f, c, o = self.ln_gates(self.gates(combined)).chunk(4, 1)
        # 输入门
        ingate = F.sigmoid(i)      # (B, H)
        # 遗忘门
        forgetgate = F.sigmoid(f)  # (B, H)
        # 输出门
        outgate = F.sigmoid(o)     # (B, H)
        # 更新细胞状态
        ncs = F.tanh(c)            # (B, H)
        cs = self.ln_c((forgetgate * cs) + (ingate * ncs))  # (B, H)
        # 更新隐藏状态
        hs = outgate * F.tanh(cs)                           # (B, H)
        return hs, cs

    def init_state(self, B, device):
        cs = torch.zeros((B, self.hidden_size), device=device)
        hs = torch.zeros((B, self.hidden_size), device=device)
        return hs, cs

class LSTMLayerNorm(nn.Module):

    def __init__(self, input_size, hidden_size):
        '''
        单层的长短期记忆网络（支持批量计算且内含层归一化）
        参数
        ----
        input_size ：int，输入数据的特征长度
        hidden_size ：int，隐藏状态的特征长度
        '''
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.lstm = LSTMLayerNormCell(self.input_size, self.hidden_size)

    def forward(self, inputs, state=None):
        '''
        向前传播
        参数
        ----
        inputs ：torch.FloatTensor
            输入数据的集合，形状为(B, T, C)，其中B表示批量大小，T表示文本长度，C表示文字特征的长度（input_size）
        state ：tuple(torch.FloatTensor, torch.FloatTensor)
            (初始的隐藏状态，初始的细胞状态)，两个状态的形状都为(B, H)，其中H表示隐藏状态的长度（hidden_size）
        返回
        ----
        hidden ：torch.FloatTensor，所有隐藏状态的集合，形状为(B, T, H)
        '''
        re = []
        B, T, C = inputs.shape
        inputs = inputs.transpose(0, 1)  # (T, B, C)
        for i in range(T):
            state = self.lstm(inputs[i], state)
            # 只记录隐藏状态，state[0]的形状为(B, H)
            re.append(state[0])
        result_tensor = torch.stack(re, dim=0)  # (T, B, H)
        return result_tensor.transpose(0, 1)    # (B, T, H)

In [ ]:
class CharLSTMLayerNorm(nn.Module):

    def __init__(self, vs):
        '''
        三层的长短期记忆网络（内嵌层归一化）
        参数
        ----
        vs ：int，字典大小
        '''
        super().__init__()
        self.emb_size = 256
        self.hidden_size = 128
        self.embedding = nn.Embedding(vs, self.emb_size)
        self.dp = nn.Dropout(0.4)
        self.lstm1 = LSTMLayerNorm(self.emb_size, self.hidden_size)
        self.lstm2 = LSTMLayerNorm(self.hidden_size, self.hidden_size)
        self.lstm3 = LSTMLayerNorm(self.hidden_size, self.hidden_size)
        self.h2o = nn.Linear(self.hidden_size, vs)

    def forward(self, x):
        '''
        向前传播
        参数
        ----
        x ：torch.LongTensor，当前字母在字典中的位置，形状为(B, T)
        返回
        ----
        output ：torch.FloatTensor，预测结果的logits，形状为(B, T, vs)
        '''
        emb = self.embedding(x)       # (B, T,  C)
        h = self.dp(self.lstm1(emb))  # (B, T,  H)
        h = self.dp(self.lstm2(h))    # (B, T,  H)
        h = self.dp(self.lstm3(h))    # (B, T,  H)
        output = self.h2o(h)          # (B, T, vs)
        return output

model_norm = CharLSTMLayerNorm(len(tok.char2ind)).to(device)

In [ ]:
l_norm = train_lstm(model_norm, optim.Adam(model_norm.parameters(), lr=learning_rate),
                    train_loader,epochs=2)

In [ ]:
# 使用模型来生成文本
begin_text = torch.tensor(tok.encode('def '), device=device).unsqueeze(0)
print(''.join(tok.decode(generate_batch(model_norm, begin_text))))